# Lecture 6: Self-supervised learning and autoencoders

## MagnaTagATune — 3-second segments

This notebook compares three autoencoder approaches on MagnaTagATune (MTAT): an unchanged baseline, deeper encoder/decoder variants, and models trained with enhanced multi-resolution STFT perceptual losses. The bottleneck compression rate is kept fixed at 31.25:1.

In [ ]:
# Install deps

!pip install torchcodec
!pip install torch
!pip install mirdata
!pip install tqdm
!pip install scikit-learn
!pip install torchaudio
!pip install matplotlib
!pip install umap-learn

In [ ]:
import os
import json
import shutil
import subprocess
from collections import OrderedDict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import umap

from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm, trange
from IPython.display import Audio

# Set to cuda or cpu, depending on the runtime
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## Dataset preparation — MagnaTagATune

The audio is divided into non-overlapping 3-second segments. Track-level splitting is performed before segment creation to prevent segments from the same recording appearing in both training and validation sets.

In [ ]:
# Mount Google Drive and define the requested results folder
from google.colab import drive
drive.mount('/content/drive')

RESULTS_DIR = "/content/drive/MyDrive/MLSM/Lab 3/results_v5"
os.makedirs(RESULTS_DIR, exist_ok=True)

# MagnaTagATune files hosted by City, University of London
MTAT_ROOT = "/content/mtat"
AUDIO_DIR = os.path.join(MTAT_ROOT, "mp3")
ANNOTATIONS_PATH = os.path.join(MTAT_ROOT, "annotations_final.csv")

os.makedirs(MTAT_ROOT, exist_ok=True)

# Download the annotations and the three parts of the audio archive only when needed
if not os.path.exists(ANNOTATIONS_PATH):
    subprocess.run([
        "wget", "-c",
        "https://mirg.city.ac.uk/datasets/magnatagatune/annotations_final.csv",
        "-O", ANNOTATIONS_PATH
    ], check=True)

if not os.path.isdir(AUDIO_DIR):
    archive_parts = ["mp3.zip.001", "mp3.zip.002", "mp3.zip.003"]
    for part in archive_parts:
        part_path = os.path.join(MTAT_ROOT, part)
        if not os.path.exists(part_path):
            subprocess.run([
                "wget", "-c",
                f"https://mirg.city.ac.uk/datasets/magnatagatune/{part}",
                "-O", part_path
            ], check=True)

    archive_path = os.path.join(MTAT_ROOT, "mp3.zip")
    with open(archive_path, "wb") as output_file:
        for part in archive_parts:
            with open(os.path.join(MTAT_ROOT, part), "rb") as input_file:
                shutil.copyfileobj(input_file, output_file)

    subprocess.run(["unzip", "-q", archive_path, "-d", MTAT_ROOT], check=True)

print("Audio directory:", AUDIO_DIR)
print("Annotations file:", ANNOTATIONS_PATH)
print("Audio directory exists:", os.path.isdir(AUDIO_DIR))

In [ ]:
# Read MTAT annotations
annotations = pd.read_csv(ANNOTATIONS_PATH, sep="\t")

# Identify metadata columns and tag columns
metadata_columns = [column for column in ["clip_id", "mp3_path"] if column in annotations.columns]
tag_columns = [column for column in annotations.columns if column not in metadata_columns]

# Keep rows with an available audio path and at least one positive tag
annotations = annotations.dropna(subset=["mp3_path"]).copy()
annotations[tag_columns] = annotations[tag_columns].apply(pd.to_numeric, errors="coerce").fillna(0)
annotations = annotations.loc[annotations[tag_columns].sum(axis=1) > 0].reset_index(drop=True)

# Use the most strongly annotated tag as the display label for UMAP
annotations["natural_label"] = annotations[tag_columns].idxmax(axis=1)

# Split by original recording, not by segment, to avoid data leakage
train_rows, val_rows = train_test_split(
    np.arange(len(annotations)),
    test_size=0.2,
    random_state=42,
    shuffle=True
)

train_annotations = annotations.iloc[train_rows].reset_index(drop=True)
val_annotations = annotations.iloc[val_rows].reset_index(drop=True)

print(f"Tracks: {len(annotations)}")
print(f"Training tracks: {len(train_annotations)}")
print(f"Validation tracks: {len(val_annotations)}")
print(f"Number of tags: {len(tag_columns)}")

In [ ]:
class MTATSegmentDataset(Dataset):
    """MagnaTagATune dataset divided into non-overlapping 3-second segments."""

    def __init__(self, annotations, audio_root, sample_rate=16000, segment_duration=3):
        self.annotations = annotations.reset_index(drop=True)
        self.audio_root = audio_root
        self.sample_rate = sample_rate
        self.segment_duration = segment_duration
        self.segment_samples = sample_rate * segment_duration
        self.orig_sample_rate = 16000

        # MTAT clips are approximately 29 seconds long.
        # Nine non-overlapping 3-second segments are used per recording.
        self.segments = []
        for track_idx in range(len(self.annotations)):
            for start_second in range(0, 27, segment_duration):
                self.segments.append((track_idx, start_second))

        # Small per-worker LRU cache avoids decoding the same MP3 for every segment.
        self.audio_cache = OrderedDict()
        self.cache_size = 8

    def __len__(self):
        return len(self.segments)

    def _load_track(self, track_idx):
        if track_idx in self.audio_cache:
            self.audio_cache.move_to_end(track_idx)
            return self.audio_cache[track_idx]

        row = self.annotations.iloc[track_idx]
        audio_path = os.path.join(MTAT_ROOT, str(row["mp3_path"]))

        if not os.path.exists(audio_path):
            raise FileNotFoundError(f"Audio file not found: {audio_path}")

        audio, sr = torchaudio.load(audio_path)

        # Convert to mono
        if audio.shape[0] > 1:
            audio = audio.mean(dim=0, keepdim=True)

        # Resample only if the stored sample rate differs from 16 kHz
        if sr != self.sample_rate:
            audio = torchaudio.transforms.Resample(sr, self.sample_rate)(audio)

        audio = audio.squeeze(0)

        # Normalize the complete recording
        max_val = torch.abs(audio).max()
        if max_val > 0:
            audio = audio / max_val

        self.audio_cache[track_idx] = audio
        if len(self.audio_cache) > self.cache_size:
            self.audio_cache.popitem(last=False)

        return audio

    def __getitem__(self, idx):
        track_idx, start_second = self.segments[idx]
        audio = self._load_track(track_idx)

        start = start_second * self.sample_rate
        end = start + self.segment_samples
        segment = audio[start:end]

        if len(segment) < self.segment_samples:
            segment = F.pad(segment, (0, self.segment_samples - len(segment)))

        row = self.annotations.iloc[track_idx]
        return {
            "audio": segment,
            "natural_labels": row["natural_label"],
            "track_id": str(row["clip_id"]) if "clip_id" in row else str(track_idx)
        }

train_dataset = MTATSegmentDataset(train_annotations, AUDIO_DIR)
val_dataset = MTATSegmentDataset(val_annotations, AUDIO_DIR)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=(device == "cuda"))
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=(device == "cuda"))

print(f"Training segments: {len(train_dataset)}")
print(f"Validation segments: {len(val_dataset)}")
print(f"Segment length: {train_dataset.segment_samples} samples ({train_dataset.segment_duration} seconds)")

# 2. Trained on a dataset — Baseline

The baseline keeps the original three convolutional encoder blocks, three pooling operations, decoder structure, batch normalization, PReLU activations, Adam optimizer and MSE objective. The input duration is changed to 3 seconds and the bottleneck is scaled proportionally to preserve the original compression rate.

In [ ]:
import torch
import torch.nn as nn

class AudioAutoencoder(nn.Module):
    """A more capable audio autoencoder featuring batch-norm, PReLu activations,
    and sharedd indices for the downsampling and upsampling layers.
    """
    def __init__(
        self,
        input_length=16000,
        bottleneck_size=64,
        kernel_size=5,
        channels=8,
        channel_mult=2,
        downsample=4,
    ):
        super().__init__()
        assert kernel_size % 2 == 1, "kernel_size should be odd for same padding"

        padding = kernel_size // 2

        # ----- Encoder -----
        self.enc_conv1 = nn.Conv1d(1, channels, kernel_size=kernel_size, padding=padding)
        self.enc_norm1 = nn.BatchNorm1d(channels)
        self.enc_act1 = nn.PReLU()
        self.pool1 = nn.MaxPool1d(2, stride=downsample, return_indices=True)

        self.enc_conv2 = nn.Conv1d(channels, channels * channel_mult, kernel_size=kernel_size, padding=padding)
        self.enc_norm2 = nn.BatchNorm1d(channels * channel_mult)
        self.enc_act2 = nn.PReLU()
        self.pool2 = nn.MaxPool1d(2, stride=downsample, return_indices=True)

        self.enc_conv3 = nn.Conv1d(channels * channel_mult, channels * channel_mult ** 2, kernel_size=kernel_size, padding=padding)
        self.enc_norm3 = nn.BatchNorm1d(channels * channel_mult ** 2)
        self.enc_act3 = nn.PReLU()
        self.pool3 = nn.MaxPool1d(2, stride=downsample, return_indices=True)

        enc_out_len = input_length // downsample ** 3 * channels * channel_mult ** 2  # 3× pooling halves length thrice
        self.enc_fc = nn.Linear(enc_out_len, bottleneck_size)

        comp_rate =  input_length / bottleneck_size
        print(f"Compression rate: {comp_rate:.1f}X")

        # ----- Decoder -----
        self.dec_fc = nn.Linear(bottleneck_size, enc_out_len)

        self.unpool3 = nn.MaxUnpool1d(2, stride=downsample)
        self.dec_deconv3 = nn.Conv1d(channels * channel_mult ** 2, channels * channel_mult, kernel_size=kernel_size, padding=padding)
        self.dec_act3 = nn.PReLU()

        self.unpool2 = nn.MaxUnpool1d(2, stride=downsample)
        self.dec_deconv2 = nn.Conv1d(channels * channel_mult, channels, kernel_size=kernel_size, padding=padding)
        self.dec_act2 = nn.PReLU()

        self.unpool1 = nn.MaxUnpool1d(2, stride=downsample)
        self.dec_deconv1 = nn.Conv1d(channels, 1, kernel_size=kernel_size, padding=padding)
        self.dec_tanh = nn.Tanh()

    def encoder(self, x):
        sizes, indices = [], []

        x = self.enc_act1(self.enc_norm1(self.enc_conv1(x)))
        sizes.append(x.size())
        x, idx1 = self.pool1(x)
        indices.append(idx1)

        x = self.enc_act2(self.enc_norm2(self.enc_conv2(x)))
        sizes.append(x.size())
        x, idx2 = self.pool2(x)
        indices.append(idx2)

        x = self.enc_act3(self.enc_norm3(self.enc_conv3(x)))
        sizes.append(x.size())
        x, idx3 = self.pool3(x)
        indices.append(idx3)
        x = x.flatten(start_dim=1)

        x = self.enc_fc(x)
        return x, indices, sizes

    # ----- Decoder -----
    def decoder(self, z, indices, sizes):
        idx1, idx2, idx3 = indices
        size1, size2, size3 = sizes

        x = self.dec_fc(z)
        x = x.view(size3[0], size3[1], -1)

        x = self.unpool3(x, idx3, output_size=size3)
        x = self.dec_act3(self.dec_deconv3(x))

        x = self.unpool2(x, idx2, output_size=size2)
        x = self.dec_act2(self.dec_deconv2(x))

        x = self.unpool1(x, idx1, output_size=size1)
        x = self.dec_tanh(self.dec_deconv1(x))

        return x

    # ----- Forward -----
    def forward(self, x):
        z, indices, sizes = self.encoder(x)
        x_r = self.decoder(z, indices, sizes)
        return x_r, z

In [ ]:
# Keep the original compression rate: 16,000 / 512 = 48,000 / 1,536 = 31.25
target_length = 48000
bottleneck_size = 1536
downsample = 2
num_epochs = 20

COMPRESSION_RATE = target_length / bottleneck_size
assert abs(COMPRESSION_RATE - 31.25) < 1e-8
print(f"Compression rate: {COMPRESSION_RATE:.2f}:1")

# Shared training/evaluation routine used by the three model families
def evaluate_model(model, loader, stft_config=None):
    model.eval()
    mse_values = []
    stft_values = []

    with torch.no_grad():
        for batch in loader:
            x = batch["audio"].unsqueeze(1).to(device)
            x_reconstr, _ = model(x)
            mse_values.append(F.mse_loss(x_reconstr, x).item())

            if stft_config is not None:
                stft_values.append(advanced_perceptual_loss(x, x_reconstr, **stft_config).item())

    return float(np.mean(mse_values)), (float(np.mean(stft_values)) if stft_values else np.nan)


def train_model(model, model_name, loss_mode="mse", stft_config=None):
    model = model.to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    history = {
        "train_mse": [],
        "val_mse": [],
        "train_stft": [],
        "val_stft": [],
        "train_objective": [],
    }

    pbar = trange(num_epochs, desc=f"Training {model_name}", unit="epoch")
    for epoch in pbar:
        model.train()
        train_mse = []
        train_stft = []
        train_objective = []

        for batch in train_loader:
            x = batch["audio"].unsqueeze(1).to(device)
            optimizer.zero_grad()

            x_reconstr, _ = model(x)
            mse_loss = criterion(x_reconstr, x)

            if loss_mode == "mse":
                loss = mse_loss
                stft_loss = torch.tensor(float("nan"), device=device)
            else:
                stft_loss = advanced_perceptual_loss(x, x_reconstr, **stft_config)
                loss = stft_config["mse_weight"] * mse_loss + stft_config["stft_weight"] * stft_loss

            loss.backward()
            optimizer.step()

            train_mse.append(mse_loss.item())
            train_objective.append(loss.item())
            if loss_mode != "mse":
                train_stft.append(stft_loss.item())

        val_mse, val_stft = evaluate_model(
            model, val_loader,
            stft_config=stft_config if loss_mode != "mse" else None
        )

        history["train_mse"].append(float(np.mean(train_mse)))
        history["val_mse"].append(val_mse)
        history["train_objective"].append(float(np.mean(train_objective)))
        history["train_stft"].append(float(np.mean(train_stft)) if train_stft else np.nan)
        history["val_stft"].append(val_stft)

        pbar.set_postfix({
            "train MSE": f"{history['train_mse'][-1]:.6f}",
            "val MSE": f"{val_mse:.6f}",
            "val STFT": f"{val_stft:.5f}" if not np.isnan(val_stft) else "n/a"
        })

    torch.save(model.state_dict(), os.path.join(RESULTS_DIR, f"{model_name}.pt"))
    with open(os.path.join(RESULTS_DIR, f"{model_name}_history.json"), "w") as file:
        json.dump(history, file, indent=2)

    return model, history


# Baseline: same architecture and training parameters as the original dataset experiment
baseline_model = AudioAutoencoder(
    input_length=target_length,
    bottleneck_size=bottleneck_size,
    downsample=downsample
)
baseline_model, baseline_history = train_model(
    baseline_model,
    "baseline",
    loss_mode="mse"
)

# 2.1 Improved — Additional convolutional layers

The improved model adds convolutional/pooling blocks to the original encoder and mirrored decoder. The loop tests multiple depths while keeping the bottleneck compression rate fixed at 31.25:1. All other training settings remain the same as the baseline.

In [ ]:
class AudioAutoencoderImproved(nn.Module):
    """Deeper version of the original autoencoder with mirrored encoder/decoder blocks."""

    def __init__(
        self,
        input_length=48000,
        bottleneck_size=1536,
        kernel_size=5,
        channels=8,
        channel_mult=2,
        downsample=2,
        extra_layers=1,
    ):
        super().__init__()
        assert kernel_size % 2 == 1, "kernel_size should be odd for same padding"

        self.num_layers = 3 + extra_layers
        self.downsample = downsample
        padding = kernel_size // 2

        self.encoder_convs = nn.ModuleList()
        self.encoder_norms = nn.ModuleList()
        self.encoder_acts = nn.ModuleList()
        self.pools = nn.ModuleList()

        self.channel_sizes = [channels * (channel_mult ** i) for i in range(self.num_layers)]

        in_channels = 1
        for out_channels in self.channel_sizes:
            self.encoder_convs.append(
                nn.Conv1d(in_channels, out_channels, kernel_size=kernel_size, padding=padding)
            )
            self.encoder_norms.append(nn.BatchNorm1d(out_channels))
            self.encoder_acts.append(nn.PReLU())
            self.pools.append(nn.MaxPool1d(2, stride=downsample, return_indices=True))
            in_channels = out_channels

        enc_out_len = (input_length // downsample ** self.num_layers) * self.channel_sizes[-1]
        self.enc_fc = nn.Linear(enc_out_len, bottleneck_size)
        self.dec_fc = nn.Linear(bottleneck_size, enc_out_len)

        self.unpools = nn.ModuleList()
        self.decoder_convs = nn.ModuleList()
        self.decoder_acts = nn.ModuleList()

        for layer_idx in reversed(range(self.num_layers)):
            in_ch = self.channel_sizes[layer_idx]
            out_ch = self.channel_sizes[layer_idx - 1] if layer_idx > 0 else 1
            self.unpools.append(nn.MaxUnpool1d(2, stride=downsample))
            self.decoder_convs.append(
                nn.Conv1d(in_ch, out_ch, kernel_size=kernel_size, padding=padding)
            )
            self.decoder_acts.append(nn.PReLU() if layer_idx > 0 else nn.Tanh())

        comp_rate = input_length / bottleneck_size
        print(f"Improved compression rate: {comp_rate:.2f}X | Extra layers: {extra_layers}")

    def encoder(self, x):
        sizes, indices = [], []

        for conv, norm, act, pool in zip(
            self.encoder_convs, self.encoder_norms, self.encoder_acts, self.pools
        ):
            x = act(norm(conv(x)))
            sizes.append(x.size())
            x, idx = pool(x)
            indices.append(idx)

        x = x.flatten(start_dim=1)
        z = self.enc_fc(x)
        return z, indices, sizes

    def decoder(self, z, indices, sizes):
        x = self.dec_fc(z)
        x = x.view(sizes[-1][0], sizes[-1][1], -1)

        for idx, size, unpool, conv, act in zip(
            reversed(indices), reversed(sizes), self.unpools,
            self.decoder_convs, self.decoder_acts
        ):
            x = unpool(x, idx, output_size=size)
            x = act(conv(x))

        return x

    def forward(self, x):
        z, indices, sizes = self.encoder(x)
        x_reconstructed = self.decoder(z, indices, sizes)
        return x_reconstructed, z

In [ ]:
improved_results = {}

# Test deeper architectures by adding one and two encoder/decoder blocks
for extra_layers in [1, 2]:
    model_name = f"improved_{extra_layers}_extra_layers"

    improved_model = AudioAutoencoderImproved(
        input_length=target_length,
        bottleneck_size=bottleneck_size,
        downsample=downsample,
        extra_layers=extra_layers
    )

    improved_model, improved_history = train_model(
        improved_model,
        model_name,
        loss_mode="mse"
    )

    improved_results[model_name] = {
        "model": improved_model,
        "history": improved_history
    }

# 3. Improve reconstruction with perceptual losses — STFT models

The original multi-scale STFT magnitude loss is retained as the first configuration. Two additional configurations extend it with spectral convergence, log-magnitude distance and a weighted multi-resolution objective. Each STFT experiment is trained independently and is also evaluated in MSE, so its reconstruction quality can be compared fairly with the baseline and improved models.

In [ ]:
def multi_scale_stft_loss(x, y, scales=[512, 1024, 2048], hop_ratio=0.25):
    """Simple implementation of the multi-scale STFT loss.
    https://arxiv.org/pdf/1910.11480
    """
    loss = 0
    for n_fft in scales:
        hop = int(n_fft * hop_ratio)
        window = torch.hann_window(n_fft, device=x.device)
        X = torch.stft(x.squeeze(), n_fft=n_fft, hop_length=hop, window=window, return_complex=True)
        Y = torch.stft(y.squeeze(), n_fft=n_fft, hop_length=hop, window=window, return_complex=True)
        loss += torch.mean(torch.abs(X.abs() - Y.abs()))
    return loss / len(scales)

In [ ]:
def advanced_perceptual_loss(
    x,
    y,
    scales=(512, 1024, 2048),
    hop_ratio=0.25,
    spectral_convergence_weight=0.0,
    log_magnitude_weight=0.0,
    magnitude_weight=1.0,
):
    """Multi-resolution STFT loss with optional spectral convergence and log-magnitude terms."""
    magnitude_losses = []
    convergence_losses = []
    log_losses = []

    x = x.squeeze(1)
    y = y.squeeze(1)

    for n_fft in scales:
        hop = int(n_fft * hop_ratio)
        window = torch.hann_window(n_fft, device=x.device)

        X = torch.stft(
            x, n_fft=n_fft, hop_length=hop, window=window,
            return_complex=True
        )
        Y = torch.stft(
            y, n_fft=n_fft, hop_length=hop, window=window,
            return_complex=True
        )

        X_mag = X.abs()
        Y_mag = Y.abs()
        magnitude_losses.append(F.l1_loss(X_mag, Y_mag))

        eps = 1e-7
        numerator = torch.linalg.vector_norm((X_mag - Y_mag).flatten(start_dim=1), dim=1)
        denominator = torch.linalg.vector_norm(X_mag.flatten(start_dim=1), dim=1).clamp_min(eps)
        convergence_losses.append((numerator / denominator).mean())

        log_losses.append(
            F.l1_loss(torch.log(X_mag + eps), torch.log(Y_mag + eps))
        )

    magnitude_loss = torch.stack(magnitude_losses).mean()
    convergence_loss = torch.stack(convergence_losses).mean()
    log_loss = torch.stack(log_losses).mean()

    return (
        magnitude_weight * magnitude_loss
        + spectral_convergence_weight * convergence_loss
        + log_magnitude_weight * log_loss
    )

In [ ]:
# STFT configurations: original multi-scale magnitude loss, then two enhanced perceptual objectives
stft_configs = {
    "stft_original": {
        "scales": (512, 1024, 2048),
        "hop_ratio": 0.25,
        "spectral_convergence_weight": 0.0,
        "log_magnitude_weight": 0.0,
        "magnitude_weight": 1.0,
        "mse_weight": 0.5,
        "stft_weight": 1.0
    },
    "stft_sc_log": {
        "scales": (512, 1024, 2048),
        "hop_ratio": 0.25,
        "spectral_convergence_weight": 1.0,
        "log_magnitude_weight": 1.0,
        "magnitude_weight": 1.0,
        "mse_weight": 0.5,
        "stft_weight": 1.0
    },
    "stft_weighted_multires": {
        "scales": (256, 512, 1024, 2048),
        "hop_ratio": 0.25,
        "spectral_convergence_weight": 1.0,
        "log_magnitude_weight": 2.0,
        "magnitude_weight": 0.5,
        "mse_weight": 0.5,
        "stft_weight": 1.0
    }
}

stft_results = {}

for model_name, stft_config in stft_configs.items():
    stft_model = AudioAutoencoder(
        input_length=target_length,
        bottleneck_size=bottleneck_size,
        downsample=downsample
    )

    stft_model, stft_history = train_model(
        stft_model,
        model_name,
        loss_mode="stft",
        stft_config=stft_config
    )

    stft_results[model_name] = {
        "model": stft_model,
        "history": stft_history,
        "config": stft_config
    }

# 4. Compare validation MSE during training

The following figure compares the validation MSE for the baseline, every improved-depth configuration and every STFT perceptual-loss configuration.

In [ ]:
all_results = {
    "baseline": {"model": baseline_model, "history": baseline_history}
}
all_results.update(improved_results)
all_results.update(stft_results)

plt.figure(figsize=(12, 7))

for model_name, result in all_results.items():
    plt.plot(
        range(1, len(result["history"]["val_mse"]) + 1),
        result["history"]["val_mse"],
        marker="o",
        markersize=3,
        label=model_name
    )

plt.title("Validation MSE during training — all model configurations")
plt.xlabel("Epoch")
plt.ylabel("Validation MSE")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "validation_mse_comparison.png"), dpi=200)
plt.show()

# 5. STFT loss versus MSE

This plot tracks the validation STFT perceptual loss against validation MSE across epochs for each STFT configuration. It helps show whether perceptual improvements are accompanied by changes in waveform reconstruction error.

In [ ]:
plt.figure(figsize=(10, 7))

for model_name, result in stft_results.items():
    history = result["history"]
    plt.plot(
        history["val_mse"],
        history["val_stft"],
        marker="o",
        label=model_name
    )

plt.title("Validation STFT perceptual loss versus validation MSE")
plt.xlabel("Validation MSE")
plt.ylabel("Validation STFT loss")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "stft_vs_mse.png"), dpi=200)
plt.show()

# 6. UMAP projections of the latent representations

A separate UMAP is generated for each trained configuration using the validation embeddings. To keep the visualization computationally manageable, up to 5,000 validation segments are sampled with a fixed random seed.

In [ ]:
def plot_umap(model, model_name, max_points=5000):
    model.eval()
    embeddings = []
    natural_labels = []
    collected = 0

    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Extracting embeddings: {model_name}"):
            x = batch["audio"].unsqueeze(1).to(device)
            _, z = model(x)

            embeddings.append(z.cpu().numpy())
            natural_labels.extend(batch["natural_labels"])
            collected += len(z)

            if collected >= max_points:
                break

    embeddings = np.concatenate(embeddings, axis=0)[:max_points]
    natural_labels = np.asarray(natural_labels)[:max_points]

    reducer = umap.UMAP(
        n_components=2,
        n_neighbors=15,
        min_dist=0.1,
        random_state=42
    )
    umap_embeddings = reducer.fit_transform(embeddings)

    unique_labels, label_indices = np.unique(natural_labels, return_inverse=True)

    plt.figure(figsize=(10, 8))
    scatter = plt.scatter(
        umap_embeddings[:, 0],
        umap_embeddings[:, 1],
        c=label_indices,
        cmap="tab20",
        s=6,
        alpha=0.75
    )

    handles, _ = scatter.legend_elements(num=min(len(unique_labels), 20))
    legend_labels = unique_labels[:len(handles)]
    plt.legend(handles, legend_labels, title="Most common MTAT tag", loc="best", fontsize=7)

    plt.title(f"UMAP of Audio Embeddings — {model_name}")
    plt.xlabel("UMAP 1")
    plt.ylabel("UMAP 2")
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, f"umap_{model_name}.png"), dpi=200)
    plt.show()


for model_name, result in all_results.items():
    plot_umap(result["model"], model_name)

In [ ]:
# Save a compact summary of the final validation metrics and all experiment settings
summary = []

for model_name, result in all_results.items():
    history = result["history"]
    row = {
        "model": model_name,
        "final_train_mse": history["train_mse"][-1],
        "final_validation_mse": history["val_mse"][-1],
        "compression_rate": COMPRESSION_RATE,
        "segment_duration_seconds": 3,
        "sample_rate": 16000
    }
    if model_name in stft_results:
        row["final_validation_stft"] = history["val_stft"][-1]
        row["stft_config"] = json.dumps(stft_results[model_name]["config"])
    else:
        row["final_validation_stft"] = np.nan
        row["stft_config"] = ""
    summary.append(row)

summary_df = pd.DataFrame(summary)
summary_df.to_csv(os.path.join(RESULTS_DIR, "training_summary.csv"), index=False)
summary_df